# 1 Load Data

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("../../Data/student_performance_data.csv")
df.drop(columns=['overall_score','student_id'], inplace=True)
df

,gender,study_hours_per_day,attendance_percentage,assignment_score,midterm_score,final_exam_score,participation_score,internet_access,extra_classes,parent_education,sleep_hours,grade
0,Male,4.54,69.98,36.47,70.70,53.10,17.96,Yes,No,Master,8.09,D
1,Female,5.26,84.80,34.25,27.92,87.17,11.29,No,Yes,Bachelor,4.73,D
2,Male,8.69,73.76,72.29,70.92,99.61,76.10,No,Yes,PhD,8.73,B
3,Male,4.06,45.00,97.63,31.73,88.85,33.55,No,No,Bachelor,8.22,C
4,Male,8.83,51.13,65.19,78.28,54.23,88.99,No,No,Bachelor,8.59,C
...,...,...,...,...,...,...,...,...,...,...,...,...
9995,Female,6.86,59.65,60.41,39.51,66.43,44.97,Yes,Yes,High School,8.56,C
9996,Male,2.60,83.62,62.45,48.96,81.40,45.11,Yes,Yes,Bachelor,4.21,C
9997,Female,1.46,95.40,67.08,51.51,87.58,65.49,Yes,No,Bachelor,4.72,B
9998,Female,7.15,78.24,97.73,46.51,75.49,61.21,No,No,Bachelor,5.28,B


In [2]:
df['grade'].value_counts()

grade
C    5073
B    2704
D    2008
A     154
F      61
Name: count, dtype: int64

In [3]:
df = df[df['grade'].isin(['C','B'])]

,gender,study_hours_per_day,attendance_percentage,assignment_score,midterm_score,final_exam_score,participation_score,internet_access,extra_classes,parent_education,sleep_hours,grade
2,Male,8.69,73.76,72.29,70.92,99.61,76.10,No,Yes,PhD,8.73,B
3,Male,4.06,45.00,97.63,31.73,88.85,33.55,No,No,Bachelor,8.22,C
4,Male,8.83,51.13,65.19,78.28,54.23,88.99,No,No,Bachelor,8.59,C
5,Female,1.79,53.16,33.61,54.26,79.79,58.23,No,No,Bachelor,5.43,C
6,Male,7.99,55.47,45.49,87.83,67.37,53.07,No,Yes,High School,5.68,C
...,...,...,...,...,...,...,...,...,...,...,...,...
9995,Female,6.86,59.65,60.41,39.51,66.43,44.97,Yes,Yes,High School,8.56,C
9996,Male,2.60,83.62,62.45,48.96,81.40,45.11,Yes,Yes,Bachelor,4.21,C
9997,Female,1.46,95.40,67.08,51.51,87.58,65.49,Yes,No,Bachelor,4.72,B
9998,Female,7.15,78.24,97.73,46.51,75.49,61.21,No,No,Bachelor,5.28,B


In [4]:
df.columns

Index(['gender', 'study_hours_per_day', 'attendance_percentage',
       'assignment_score', 'midterm_score', 'final_exam_score',
       'participation_score', 'internet_access', 'extra_classes',
       'parent_education', 'sleep_hours', 'grade'],
      dtype='str')

##### ohe -> 'gender'
##### ordinal -> 'parent_education'
##### Label -> 'internet_access', 'extra_classes'
##### scaling_transform -> 'study_hours_per_day', 'attendance_percentage','assignment_score', 'participation_score',midterm_score', 'final_exam_score', 'sleep_hours'



In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder, StandardScaler, PowerTransformer

scaling_transform = Pipeline([
    ("scl", StandardScaler()),
    ('power', PowerTransformer())
])

preprocessor = ColumnTransformer(transformers=[
    ("ohe", OneHotEncoder(sparse_output=False,handle_unknown='ignore'),['gender']),
    ("ord", OrdinalEncoder(categories=[['High School','Bachelor','Master','PhD']]), ['parent_education']),
    ("ord_bin", OrdinalEncoder(), ['internet_access', 'extra_classes']),
    ("scaling_transform", scaling_transform, ['study_hours_per_day', 'attendance_percentage','assignment_score', 'midterm_score', 'final_exam_score','participation_score', 'sleep_hours'])
    
],remainder='passthrough')

In [6]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:,:-1], df.iloc[:,-1], test_size=0.2, random_state=42)

In [7]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [8]:
lb = LabelEncoder()
y_train = lb.fit_transform(y_train)
y_test = lb.transform(y_test)

In [9]:
X_train

array([[ 0.        ,  1.        ,  2.        , ..., -0.71443428,
        -1.00338143,  0.26623556],
       [ 1.        ,  0.        ,  2.        , ..., -1.3161121 ,
        -0.7510872 ,  0.74189396],
       [ 1.        ,  0.        ,  2.        , ..., -1.00509617,
         0.60925752,  0.37632317],
       ...,
       [ 1.        ,  0.        ,  2.        , ...,  1.46435238,
        -0.94723832, -1.29377937],
       [ 1.        ,  0.        ,  1.        , ..., -0.00419386,
        -0.54036775,  0.1082426 ],
       [ 1.        ,  0.        ,  0.        , ...,  0.22430691,
        -0.39103647, -0.540795  ]], shape=(8000, 12))

In [10]:
import torch
X_train = torch.from_numpy(X_train.astype(np.float32))
X_test = torch.from_numpy(X_test.astype(np.float32))
y_train = torch.from_numpy(np.asarray(y_train))   # Series - > Dataframe
y_test = torch.from_numpy(np.asarray(y_test))

# 3. Build Model

In [11]:
import torch.nn as nn
class MySimplePerceptron(nn.Module):
    def __init__(self, X_train):
        super().__init__()
        self.linear = nn.Linear(X_train.shape[1], 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, X_train):
        out = self.linear(X_train)
        return self.sigmoid(out)
    
    def loss_function(self, y_pred, y_train):
        epsilon = 1e-8
        y_pred = torch.clamp(y_pred, epsilon, 1-epsilon)
        loss = (  -y_train*torch.log(y_pred) - (1-y_train)*torch.log(1-y_pred) ).mean()
        return loss

# 4. Train Model

In [14]:
def train_model(learning_rate = 0.1,epochs = 100):
    model = MySimplePerceptron(X_train)
    
    for epoch in range(epochs):
        # forward propagation
        y_pred = model.forward(X_train)
        
        # loss calculate
        loss = model.loss_function(y_pred, y_train)
        
        # backpropagation
        loss.backward()
        
        # update weight and bias
        with torch.no_grad():
            model.linear.weight -= (learning_rate * model.linear.weight.grad)
            model.linear.bias -= (learning_rate * model.linear.bias.grad)
            
        # reinitialize gradient
        model.linear.weight.grad.zero_()
        model.linear.bias.grad.zero_()
        
        print(f"Epoch: {epoch+1}, Loss: {loss.item()}")
        
    return model

In [15]:
model = train_model(epochs=100, learning_rate=0.01)

Epoch: 1, Loss: 1.4294708967208862
Epoch: 2, Loss: 1.330317735671997
Epoch: 3, Loss: 1.233304500579834
Epoch: 4, Loss: 1.1384210586547852
Epoch: 5, Loss: 1.0456494092941284
Epoch: 6, Loss: 0.9549660086631775
Epoch: 7, Loss: 0.8663405179977417
Epoch: 8, Loss: 0.7797369360923767
Epoch: 9, Loss: 0.6951145529747009
Epoch: 10, Loss: 0.6124283075332642
Epoch: 11, Loss: 0.5316289663314819
Epoch: 12, Loss: 0.45266425609588623
Epoch: 13, Loss: 0.37547963857650757
Epoch: 14, Loss: 0.30001845955848694
Epoch: 15, Loss: 0.22622261941432953
Epoch: 16, Loss: 0.1540330946445465
Epoch: 17, Loss: 0.08339071273803711
Epoch: 18, Loss: 0.014236039482057095
Epoch: 19, Loss: -0.05348972976207733
Epoch: 20, Loss: -0.11984477937221527
Epoch: 21, Loss: -0.184886634349823
Epoch: 22, Loss: -0.24867096543312073
Epoch: 23, Loss: -0.31125250458717346
Epoch: 24, Loss: -0.3726842403411865
Epoch: 25, Loss: -0.4330175817012787
Epoch: 26, Loss: -0.4923021197319031
Epoch: 27, Loss: -0.5505858063697815
Epoch: 28, Loss: -0.

# 5. Evaluate Model

In [16]:
def evaluate_model(threshold=0.5):
    with torch.no_grad():  
        y_pred = model.forward(X_test)
        y_pred = (y_pred > threshold).float()
        
        accuracy = (y_test == y_pred).float().mean()
        print(f"Accuracy: {accuracy}")

In [17]:
evaluate_model(threshold=0.5)

Accuracy: 0.27399998903274536
